# YOLO11 Face Detector Fine-Tuning

This notebook is designed for Google Colab and fine-tunes **YOLO11** for **face-only detection**.

Recommended dataset:

- `WIDERFace`

This notebook supports the full flow inside Colab:

1. download WIDERFace into Google Drive
2. inspect and resolve the extracted structure
3. convert WIDERFace annotations into YOLO format
4. generate `data.yaml`
5. fine-tune `yolo11n.pt`
6. export the best checkpoint back to Drive


## 1. Runtime

In Colab, set `Runtime -> Change runtime type -> GPU` before training.

## 2. WIDERFace structure this notebook expects

The preprocessing in this notebook is built around the common WIDERFace structure:

```text
WIDERFACE_ROOT/
  WIDER_train/
    images/
      <event_name>/<image>.jpg
  WIDER_val/
    images/
      <event_name>/<image>.jpg
  wider_face_split/
    wider_face_train_bbx_gt.txt
    wider_face_val_bbx_gt.txt
```

The annotation text files list an image path, a face count, and then bounding box rows. This notebook converts those annotations into YOLO labels with one class only:

- `0: face`


## 3. Download WIDERFace into Google Drive

This notebook supports downloading a Kaggle mirror of WIDERFace directly into Google Drive.

Before running the download cells:

1. Create a Kaggle account
2. Go to `Kaggle -> Account -> API -> Create New Token`
3. Upload the downloaded `kaggle.json` file when prompted below

If you already have WIDERFace in Drive, you can skip the download cells and point `RAW_WIDERFACE_ROOT` to that extracted folder.


In [ ]:
!pip install -q ultralytics kaggle PyYAML

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

In [ ]:
# Upload your Kaggle API token here when running in Colab.
# Skip this cell if you already set up ~/.kaggle/kaggle.json in the current runtime.
uploaded = files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!ls -la ~/.kaggle

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

import yaml
from PIL import Image
from ultralytics import YOLO


In [ ]:
# Update these paths before running training.
DOWNLOAD_WIDERFACE = True
KAGGLE_DATASET_SLUG = 'mksaad/wider-face-a-face-detection-benchmark'
DOWNLOAD_DIR = '/content/drive/MyDrive/WIDERFACE_download'
RAW_WIDERFACE_ROOT = '/content/drive/MyDrive/WIDERFACE_download/WIDERFACE'
YOLO_DATASET_ROOT = '/content/drive/MyDrive/widerface_yolo'
PROJECT_DIR = '/content/drive/MyDrive/yolo11_face_runs'
RUN_NAME = 'yolo11_face_finetune'
BASE_MODEL = 'yolo11n.pt'
EPOCHS = 50
IMAGE_SIZE = 640
BATCH = 16
DEVICE = 0
GENERATE_YOLO_DATASET = True

download_dir = Path(DOWNLOAD_DIR)
raw_widerface_root = Path(RAW_WIDERFACE_ROOT)
yolo_root = Path(YOLO_DATASET_ROOT)
project_dir = Path(PROJECT_DIR)

download_dir.mkdir(parents=True, exist_ok=True)
yolo_root.mkdir(parents=True, exist_ok=True)
project_dir.mkdir(parents=True, exist_ok=True)

print('download_dir:', download_dir)
print('raw_widerface_root:', raw_widerface_root)
print('yolo_root:', yolo_root)
print('project_dir:', project_dir)

## 4. Download and extract WIDERFace

In [ ]:
if DOWNLOAD_WIDERFACE:
    zip_path = download_dir / 'wider-face-a-face-detection-benchmark.zip'
    if not zip_path.exists():
        !kaggle datasets download -d {KAGGLE_DATASET_SLUG} -p {DOWNLOAD_DIR}
    else:
        print(f'Using existing zip: {zip_path}')

    extract_target = download_dir / 'WIDERFACE'
    if not extract_target.exists():
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_target)
    else:
        print(f'Using existing extracted folder: {extract_target}')

    print('Top-level extracted contents:')
    for p in sorted(extract_target.iterdir()):
        print('-', p.name)
else:
    print('Skipping WIDERFace download and extraction')

## 5. Resolve the actual WIDERFace root

Some archives add an extra nested folder. This cell tries to find the folder that actually contains the split files and image directories.

In [ ]:
split_candidates = list(download_dir.rglob('wider_face_train_bbx_gt.txt')) if DOWNLOAD_WIDERFACE else []
if split_candidates:
    raw_widerface_root = split_candidates[0].parent.parent.parent.parent
    print('Detected WIDERFace root:', raw_widerface_root)
else:
    print('Could not auto-detect split file. Using configured RAW_WIDERFACE_ROOT:', raw_widerface_root)

print('train split exists:', (raw_widerface_root / 'wider_face_split' / 'wider_face_split' / 'wider_face_train_bbx_gt.txt').exists())
print('val split exists:', (raw_widerface_root / 'wider_face_split' / 'wider_face_split' / 'wider_face_val_bbx_gt.txt').exists())
print('train images exists:', (raw_widerface_root / 'WIDER_train' / 'images').exists())
print('val images exists:', (raw_widerface_root / 'WIDER_val' / 'images').exists())
print('nested train images exists:', (raw_widerface_root / 'WIDER_train' / 'WIDER_train' / 'images').exists())
print('nested val images exists:', (raw_widerface_root / 'WIDER_val' / 'WIDER_val' / 'images').exists())

## 6. Convert WIDERFace to YOLO format

This cell creates the YOLO dataset structure:

```text
widerface_yolo/
  images/
    train/
    val/
  labels/
    train/
    val/
  data.yaml
```

In [ ]:
def ensure_yolo_dirs(root: Path) -> None:
    for rel in ['images/train', 'images/val', 'labels/train', 'labels/val']:
        (root / rel).mkdir(parents=True, exist_ok=True)

def convert_bbox_to_yolo(box, image_width, image_height):
    x, y, w, h = box
    x_center = (x + w / 2.0) / image_width
    y_center = (y + h / 2.0) / image_height
    width = w / image_width
    height = h / image_height
    return x_center, y_center, width, height

def resolve_wider_image_root(raw_root: Path, split: str) -> Path:
    split_title = split.capitalize()
    split_lower = split.lower()
    split_upper = split.upper()
    candidates = [
        raw_root / f'WIDER_{split_title}' / 'images',
        raw_root / f'WIDER_{split_title}' / f'WIDER_{split_title}' / 'images',
        raw_root / f'WIDER_{split_lower}' / 'images',
        raw_root / f'WIDER_{split_lower}' / f'WIDER_{split_lower}' / 'images',
        raw_root / f'WIDER_{split_upper}' / 'images',
        raw_root / f'WIDER_{split_upper}' / f'WIDER_{split_upper}' / 'images',
        raw_root / f'wider_{split}' / 'images',
        raw_root / f'wider_{split}',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not resolve image root for split={split}. Tried: {candidates}')

def parse_wider_annotation_file(annotation_path: Path):
    lines = annotation_path.read_text().splitlines()
    i = 0
    records = []
    while i < len(lines):
        image_rel = lines[i].strip()
        i += 1
        face_count = int(lines[i].strip())
        i += 1
        boxes = []
        for _ in range(face_count):
            parts = [int(v) for v in lines[i].strip().split()]
            i += 1
            x, y, w, h = parts[:4]
            if w > 0 and h > 0:
                boxes.append((x, y, w, h))
        records.append((image_rel, boxes))
    return records

def convert_widerface_split(raw_root: Path, split_name: str, annotation_file: Path, image_root: Path, yolo_root: Path) -> int:
    records = parse_wider_annotation_file(annotation_file)
    converted = 0
    for image_rel, boxes in records:
        source_image = image_root / image_rel
        if not source_image.exists():
            continue

        event_name = Path(image_rel).parent.name
        image_name = Path(image_rel).name
        target_image_dir = yolo_root / 'images' / split_name / event_name
        target_label_dir = yolo_root / 'labels' / split_name / event_name
        target_image_dir.mkdir(parents=True, exist_ok=True)
        target_label_dir.mkdir(parents=True, exist_ok=True)

        target_image = target_image_dir / image_name
        target_label = target_label_dir / f'{Path(image_name).stem}.txt'

        if not target_image.exists():
            shutil.copy2(source_image, target_image)

        with Image.open(source_image) as img:
            image_width, image_height = img.size

        yolo_lines = []
        for box in boxes:
            x_center, y_center, width, height = convert_bbox_to_yolo(box, image_width, image_height)
            yolo_lines.append(f'0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}')

        target_label.write_text('\n'.join(yolo_lines))
        converted += 1

    return converted

def write_data_yaml(yolo_root: Path) -> Path:
    data = {
        'path': str(yolo_root),
        'train': 'images/train',
        'val': 'images/val',
        'names': {0: 'face'},
    }
    yaml_path = yolo_root / 'data.yaml'
    with yaml_path.open('w') as f:
        yaml.safe_dump(data, f, sort_keys=False)
    return yaml_path

if GENERATE_YOLO_DATASET:
    ensure_yolo_dirs(yolo_root)
    train_annotations = raw_widerface_root / 'wider_face_split' / 'wider_face_split' / 'wider_face_train_bbx_gt.txt'
    val_annotations = raw_widerface_root / 'wider_face_split' / 'wider_face_split' / 'wider_face_val_bbx_gt.txt'
    train_images = resolve_wider_image_root(raw_widerface_root, 'train')
    val_images = resolve_wider_image_root(raw_widerface_root, 'val')

    print('resolved train image root:', train_images)
    print('resolved val image root:', val_images)

    train_count = convert_widerface_split(raw_widerface_root, 'train', train_annotations, train_images, yolo_root)
    val_count = convert_widerface_split(raw_widerface_root, 'val', val_annotations, val_images, yolo_root)
    data_yaml = write_data_yaml(yolo_root)

    print('Converted train images:', train_count)
    print('Converted val images:', val_count)
    print('data.yaml:', data_yaml)
else:
    data_yaml = yolo_root / 'data.yaml'
    print('Skipping dataset generation. Expecting existing YOLO dataset at:', yolo_root)

## 7. Inspect the generated `data.yaml`

In [ ]:
print('data yaml exists:', data_yaml.exists())
with open(data_yaml, 'r') as f:
    data_cfg = yaml.safe_load(f)
print(data_cfg)

## 8. Fine-tune YOLO11 for face detection

In [ ]:
model = YOLO(BASE_MODEL)

results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    device=DEVICE,
    project=str(project_dir),
    name=RUN_NAME,
    pretrained=True,
    patience=15,
    close_mosaic=10,
    save=True,
    verbose=True,
)

print(results)

## 9. Validate the best checkpoint

In [ ]:
best_weights = project_dir / RUN_NAME / 'weights' / 'best.pt'
print('best weights exists:', best_weights.exists(), best_weights)

best_model = YOLO(str(best_weights))
metrics = best_model.val(data=str(data_yaml), imgsz=IMAGE_SIZE, device=DEVICE)
print(metrics)

## 10. Export the checkpoint for this repo

In [ ]:
export_path = Path('/content/drive/MyDrive/yolo_face.pt')
shutil.copy2(best_weights, export_path)
print('exported checkpoint to', export_path)

## After training

Download `yolo_face.pt` and place it in:

- `models/detection/`

Then the repo will use the fine-tuned YOLO face detector instead of falling back to OpenCV.